In [2]:
import nltk
import torch

In [5]:
import os

In [6]:
os.getcwd()

'C:\\Users\\njpar\\Desktop\\Northeastern\\Third Year\\Spring25\\CS4120\\project\\playground'

In [4]:
from project.repo.shake-z.src.data.data_utils import load_data, create_dataloaders
from src.models.lstm import Encoder, Decoder, LSTMModel
from src.evaluation.evaluation_metrics import bleu, chrf
from src.utils.config import CONFIG
from utils import save_results

SyntaxError: invalid syntax (3467342370.py, line 1)

## DATA UTILS

In [105]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


def load_slang_dataset(csv_path):
    """
    Load the Gen Z slang dataset.
    Expects columns: 'slang', 'description', 'example', 'context'.
    We treat 'description' as the source (standard English) and 'slang' as the target.
    """
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["Description", "Slang"])
    # Build (source, target) pairs
    pairs = list(
        zip(df["Description"].astype(str).tolist(), df["Slang"].astype(str).tolist())
    )
    return pairs


def load_parallel_dataset(csv_path):
    """
    Load parallel Shakespeare→GenZ dataset.
    Expects columns: 'source', 'target'.
    """
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["source", "target"])
    return list(zip(df["source"].astype(str), df["target"].astype(str)))


def split_data(pairs, test_size=0.1, val_size=0.1, random_state=42):
    """
    Split a list of (source, target) pairs into train, val, and test sets.
    """
    train_val, test = train_test_split(
        pairs, test_size=test_size, random_state=random_state
    )
    # Compute val size relative to the remaining data
    val_relative = val_size / (1 - test_size)
    train, val = train_test_split(
        train_val, test_size=val_relative, random_state=random_state
    )
    return train, val, test


class TranslationDataset(Dataset):
    """
    PyTorch Dataset that tokenizes on the fly.

    Each item is a dict with:
      - input_ids:         LongTensor [max_length]
      - attention_mask:    LongTensor [max_length]
      - labels:            LongTensor [max_length]  (with pad tokens masked to -100)
    """

    def __init__(self, data_pairs, tokenizer, max_length=50):
        """
        Args:
            data_pairs (List[Tuple[str,str]]): list of (source, target) strings
            tokenizer (PreTrainedTokenizer): e.g. T5Tokenizer
            max_length (int): maximum sequence length for both source & target
        """
        self.pairs = data_pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        # encode source
        enc = self.tokenizer(
            src,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # encode target
        dec = self.tokenizer(
            tgt,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # prepare labels (mask pad tokens as -100 so they’re ignored in loss)
        labels = dec.input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "labels": labels.squeeze(0),
        }


def create_dataloaders(
    train_pairs,
    val_pairs,
    test_pairs,
    tokenizer,
    max_length,
    batch_size=32,
    shuffle=True,
):
    """
    Build PyTorch DataLoaders for train/val/test splits.

    Args:
        train_pairs (List[Tuple[str,str]]): (source, target) for training.
        val_pairs   (List[Tuple[str,str]]): for validation.
        test_pairs  (List[Tuple[str,str]]): for testing.
        tokenizer   (PreTrainedTokenizer): e.g. T5Tokenizer.
        max_length  (int): max sequence length for both source & target.
        batch_size  (int): batch size.
        shuffle     (bool): whether to shuffle the train split.

    Returns:
        train_loader, val_loader, test_loader: three DataLoader objects,
        each yielding dicts with keys "input_ids", "attention_mask", "labels".
    """
    # wrap each split in our on‑the‑fly tokenizing Dataset
    train_ds = TranslationDataset(train_pairs, tokenizer, max_length)
    val_ds = TranslationDataset(val_pairs, tokenizer, max_length)
    test_ds = TranslationDataset(test_pairs, tokenizer, max_length)

    # default collate will batch the fixed-size tensors
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


def load_data(config):
    path = config["data_path"]
    dataset = config.get("dataset", "shakez")
    if dataset == "shakez":
        csv_path = os.path.join(path, "mappings/shakez.csv")
        pairs = load_parallel_dataset(csv_path)
    elif dataset == "sonnetz":
        csv_path = os.path.join(path, "mappings/sonnetz.csv")
        pairs = load_parallel_dataset(csv_path)
    else:
        raise ValueError(f"Unknown dataset: {dataset}")
    test_size = config.get("test_size", 0.1)
    val_size = config.get("val_size", 0.1)
    return split_data(pairs, test_size=test_size, val_size=val_size)


# EVALUATION METRICS

In [11]:
import nltk
from nltk.translate.bleu_score import corpus_bleu
from sacrebleu.metrics import CHRF

# Uncomment the following line if you haven't already downloaded the required tokenizer data
# nltk.download('punkt')


def bleu(model, dataset):
    """
    Compute the corpus BLEU score for a given model on a dataset.

    Args:
        model: A model with a translate() method that takes a source sentence as input
               and returns a generated translation.
        dataset: A list of tuples in the form (source_sentence, reference_translation).

    Returns:
        bleu_score: A float representing the corpus BLEU score.
    """
    references = []
    hypotheses = []

    for source, reference in dataset:
        # Generate model output
        hypothesis = model.translate(source)

        # Tokenize both the model's output and the reference translation
        hyp_tokens = nltk.word_tokenize(hypothesis.lower())
        ref_tokens = nltk.word_tokenize(reference.lower())

        hypotheses.append(hyp_tokens)
        references.append([ref_tokens])

    bleu_score = corpus_bleu(references, hypotheses)
    return bleu_score


def chrf(model, dataset):
    """
    Returns the corpus-level chrF score.
    """
    chrf = CHRF()
    hyps = []
    refs = []
    for src, ref in dataset:
        hyps.append(model.translate(src))
        refs.append([ref])
    return chrf.corpus_score(hyps, refs).score


def evaluate(model, dataset, verbose=False):
    """
    Evaluate a model or translations by computing both BLEU and chrF scores.

    This function is a convenient wrapper that calls the existing `bleu` and `chrf`
    functions, passing along any additional keyword arguments.
    """

    # Call the existing bleu and chrf functions with the provided parameters
    bleu_score = bleu(model, dataset)
    chrf_score = chrf(model, dataset)
    if verbose:
        print("Model Evaluation:")
        print(f"  BLEU:      {bleu_score:.2f}")
        print(f"  chrF:      {chrf_score:.2f}")

    return bleu_score, chrf_score


if __name__ == "__main__":
    # Sample usage with a dummy model and dataset for testing purposes
    class DummyModel:
        def translate(self, text):
            # Dummy translation: return the input text unchanged
            return text

    # Create a dummy model instance
    dummy_model = DummyModel()

    # Dummy dataset: list of (source, reference) sentence pairs
    dummy_dataset = [
        ("This is a test.", "This is a test."),
        ("Another example sentence.", "Another example sentence."),
    ]

    # Compute and print BLEU score with the smoothing function
    score = evaluate(dummy_model, dummy_dataset)
    print("BLEU/chrF scores:", score)


BLEU/chrF scores: (1.0, 100.0)


# CONFIG

In [12]:
CONFIG = {
    ### Data paths
    #
    "data_path": "data/processed/",  # Path to your preprocessed dataset
    "raw_data_path": "data/raw/",  # Optional: path for raw data files
    "dataset": "shakez",
    #
    #
    ### Model hyperparameters for N-gram model
    #
    "ngram_n": 5,  # n in n-gram (e.g., 3 for trigram)
    #
    #
    ### Training parameters common to neural models
    #
    "batch_size": 32,
    "learning_rate": 0.01,
    "num_epochs": 3,  # Number of training epochs
    "warmup_steps": 100,  # Used for transformer scheduler
    # Sequence processing
    "max_seq_length": 50,  # Maximum sequence length for input/output
    # Additional configurations for model-specific parameters
    "lstm_hidden_size": 256,  # Hidden state size for LSTM model
    "lstm_num_layers": 2,  # Number of layers for LSTM model
    "dropout": 0.5,  # Dropout rate for regularization
    #
    #
    ### Transformer specific parameters (if fine-tuning a pre-trained model)
    #
    "pretrained_model_name": "t5-small",  # Name of the pre-trained model from Hugging Face
    "transformer_max_length": 50,  # Max token length for transformer inputs/outputs
    # Logging and checkpointing
    "log_interval": 100,  # How often to log training progress (in batches)
    "checkpoint_dir": "checkpoints/",  # Directory to save model checkpoints
}

if __name__ == "__main__":
    # Quick test to print configuration values
    for key, value in CONFIG.items():
        print(f"{key}: {value}")


data_path: data/processed/
raw_data_path: data/raw/
dataset: shakez
ngram_n: 5
batch_size: 32
learning_rate: 0.01
num_epochs: 3
warmup_steps: 100
max_seq_length: 50
lstm_hidden_size: 256
lstm_num_layers: 2
dropout: 0.5
pretrained_model_name: t5-small
transformer_max_length: 50
log_interval: 100
checkpoint_dir: checkpoints/


# OLD MODEL

In [8]:
import torch
import torch.nn as nn
import random


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, num_layers, dropout):
        """
        Encoder using LSTM.

        Args:
            input_dim (int): Size of the source vocabulary.
            emb_dim (int): Embedding dimension.
            hid_dim (int): Hidden state dimension.
            num_layers (int): Number of LSTM layers.
            dropout (float): Dropout rate.
        """
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, hid_dim, num_layers=num_layers, dropout=dropout, batch_first=True
        )

    def forward(self, src):
        # src shape: [batch_size, src_len]
        embedded = self.embedding(src)  # [batch_size, src_len, emb_dim]
        outputs, (hidden, cell) = self.lstm(embedded)
        # We return the final hidden and cell states to be used by the decoder.
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, num_layers, dropout):
        """
        Decoder using LSTM.

        Args:
            output_dim (int): Size of the target vocabulary.
            emb_dim (int): Embedding dimension.
            hid_dim (int): Hidden state dimension.
            num_layers (int): Number of LSTM layers.
            dropout (float): Dropout rate.
        """
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, hid_dim, num_layers=num_layers, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        # input shape: [batch_size] -> we add a time dimension
        input = input.unsqueeze(1)  # [batch_size, 1]
        embedded = self.embedding(input)  # [batch_size, 1, emb_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output shape: [batch_size, 1, hid_dim] -> squeeze to [batch_size, hid_dim]
        prediction = self.fc_out(output.squeeze(1))  # [batch_size, output_dim]
        return prediction, hidden, cell


class LSTMModel(nn.Module):
    def __init__(self, encoder, decoder):
        """
        Wrapper model that ties the encoder and decoder together.

        Args:
            encoder (nn.Module): The encoder model.
            decoder (nn.Module): The decoder model.
        """
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Forward pass through the sequence-to-sequence model using teacher forcing.

        Args:
            src (Tensor): Source tensor of shape [batch_size, src_len].
            trg (Tensor): Target tensor of shape [batch_size, trg_len].
            teacher_forcing_ratio (float): Probability of using teacher forcing.

        Returns:
            outputs (Tensor): Predicted outputs of shape [batch_size, trg_len, output_dim].
        """
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.embedding.num_embeddings

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size)

        # Encode the source sequence
        hidden, cell = self.encoder(src)

        # First input to the decoder is the <sos> token (assumed to be at index 0)
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            # Decide whether to use teacher forcing: feed the actual target as the next input, or use model's prediction.
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)  # Get the highest scoring token from predictions

            input = trg[:, t] if teacher_force else top1

        return outputs

    def translate(self, src_sentence, src_field, trg_field, max_len=50):
        """
        Translate a single sentence from source to target using greedy decoding.

        Args:
            src_sentence (str): Input sentence (standard text).
            src_field: Object with attributes: init_token, eos_token, and a vocab mapping (stoi/itos) for the source.
            trg_field: Object with attributes: init_token, eos_token, and a vocab mapping (stoi/itos) for the target.
            max_len (int): Maximum length for the generated sentence.

        Returns:
            str: The translated sentence in target language style.
        """
        self.eval()
        tokens = src_sentence.split()
        tokens = [src_field.init_token] + tokens + [src_field.eos_token]
        src_indices = [src_field.vocab.stoi[token] for token in tokens]
        src_tensor = torch.LongTensor(src_indices).unsqueeze(0)

        with torch.no_grad():
            hidden, cell = self.encoder(src_tensor)

        trg_indices = [trg_field.vocab.stoi[trg_field.init_token]]
        for _ in range(max_len):
            trg_tensor = torch.LongTensor([trg_indices[-1]])
            with torch.no_grad():
                output, hidden, cell = self.decoder(trg_tensor, hidden, cell)
            pred_token = output.argmax(1).item()
            trg_indices.append(pred_token)
            if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
                break

        # Convert indices back to words and remove special tokens
        trg_tokens = [trg_field.vocab.itos[i] for i in trg_indices]
        return " ".join(trg_tokens[1:-1])


# test block
if __name__ == "__main__":
    # Define dummy vocabulary and field objects for testing.
    class DummyVocab:
        def __init__(self):
            # Simple mapping for demonstration
            self.stoi = {
                "<s>": 0,
                "</s>": 1,
                "hello": 2,
                "world": 3,
                "how": 4,
                "are": 5,
                "you": 6,
            }
            self.itos = {i: s for s, i in self.stoi.items()}

    class DummyField:
        def __init__(self):
            self.init_token = "<s>"
            self.eos_token = "</s>"
            self.vocab = DummyVocab()

    src_field = DummyField()
    trg_field = DummyField()

    # Hyperparameters for dummy model
    INPUT_DIM = len(src_field.vocab.stoi)
    OUTPUT_DIM = len(trg_field.vocab.stoi)
    EMB_DIM = 16
    HID_DIM = 32
    NUM_LAYERS = 1
    DROPOUT = 0.1

    # Instantiate encoder, decoder, and the seq2seq model
    encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT)
    decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT)
    model = LSTMModel(encoder, decoder)

    # Test the translate() method with a dummy input
    sample_input = "hello world"
    print("Input:", sample_input)
    translation = model.translate(sample_input, src_field, trg_field)
    print("Translation:", translation)


Input: hello world
Translation: hello


C:\Users\njpar\anaconda3\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(


# (OLD) NEW LSTM MODEL

In [136]:
import torch
import torch.nn as nn
import random


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, num_layers, dropout):
        """
        Encoder using LSTM with attention mask support.

        Args:
            input_dim (int): Size of the source vocabulary.
            emb_dim (int): Embedding dimension.
            hid_dim (int): Hidden state dimension.
            num_layers (int): Number of LSTM layers.
            dropout (float): Dropout rate.
        """
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, hid_dim, num_layers=num_layers, dropout=dropout, batch_first=True
        )

    def forward(self, src, attention_mask=None):
        """
        Forward pass through the LSTM encoder.

        Args:
            src (Tensor): Input tensor [batch_size, src_len].
            attention_mask (Tensor, optional): Binary mask [batch_size, src_len] (1 = keep, 0 = pad).

        Returns:
            hidden (Tensor): Final hidden state from LSTM.
            cell (Tensor): Final cell state from LSTM.
        """
        embedded = self.embedding(src)  # [batch_size, src_len, emb_dim]

        if attention_mask is not None:
            lengths = attention_mask.sum(dim=1).cpu()  # [batch_size]
            # Pack padded sequence
            packed = nn.utils.rnn.pack_padded_sequence(
                embedded, lengths, batch_first=True, enforce_sorted=False
            )
            packed_outputs, (hidden, cell) = self.lstm(packed)
        else:
            # No mask — assume all inputs are the same length
            _, (hidden, cell) = self.lstm(embedded)

        return hidden, cell



class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, num_layers, dropout):
        """
        Decoder using LSTM.

        Args:
            output_dim (int): Size of the target vocabulary.
            emb_dim (int): Embedding dimension.
            hid_dim (int): Hidden state dimension.
            num_layers (int): Number of LSTM layers.
            dropout (float): Dropout rate.
        """
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim, hid_dim, num_layers=num_layers, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        # input shape: [batch_size] -> we add a time dimension
        input = input.unsqueeze(1)  # [batch_size, 1]
        embedded = self.embedding(input)  # [batch_size, 1, emb_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output shape: [batch_size, 1, hid_dim] -> squeeze to [batch_size, hid_dim]
        prediction = self.fc_out(output.squeeze(1))  # [batch_size, output_dim]
        return prediction, hidden, cell


class LSTMModel(nn.Module):
    def __init__(self, encoder, decoder):
        """
        Wrapper model that ties the encoder and decoder together.
        """
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input_ids, labels=None, attention_mask=None, teacher_forcing_ratio=0.5):
        """
        Forward pass for dictionary-style batch input with attention masking.

        Args:
            input_ids (Tensor): Source input tensor [batch_size, src_len].
            labels (Tensor): Target tensor [batch_size, trg_len].
            attention_mask (Tensor, optional): Mask for input_ids (1 = real token, 0 = padding).
            teacher_forcing_ratio (float): Probability to use teacher forcing.

        Returns:
            Tensor: Model output [batch_size, trg_len, vocab_size].
        """
        src = input_ids
        trg = labels

        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.embedding.num_embeddings

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=src.device)

        # Encode the source sequence (pass in the attention mask if needed)
        hidden, cell = self.encoder(src, attention_mask=0)

        # First decoder input = <sos> tokens (first column of trg)
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[:, t] if teacher_force else top1

        return outputs

    def translate(self, src_sentence, src_field, trg_field, max_len=50):
        """
        Translate a single sentence from source to target using greedy decoding.

        Args:
            src_sentence (str): Input sentence (standard text).
            src_field: Object with attributes: init_token, eos_token, and a vocab mapping (stoi/itos) for the source.
            trg_field: Object with attributes: init_token, eos_token, and a vocab mapping (stoi/itos) for the target.
            max_len (int): Maximum length for the generated sentence.

        Returns:
            str: The translated sentence in target language style.
        """
        self.eval()
        tokens = src_sentence.split()
        tokens = [src_field.init_token] + tokens + [src_field.eos_token]
        src_indices = [src_field.vocab.stoi[token] for token in tokens]
        src_tensor = torch.LongTensor(src_indices).unsqueeze(0)

        with torch.no_grad():
            hidden, cell = self.encoder(src_tensor)

        trg_indices = [trg_field.vocab.stoi[trg_field.init_token]]
        for _ in range(max_len):
            trg_tensor = torch.LongTensor([trg_indices[-1]])
            with torch.no_grad():
                output, hidden, cell = self.decoder(trg_tensor, hidden, cell)
            pred_token = output.argmax(1).item()
            trg_indices.append(pred_token)
            if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
                break

        # Convert indices back to words and remove special tokens
        trg_tokens = [trg_field.vocab.itos[i] for i in trg_indices]
        return " ".join(trg_tokens[1:-1])



# test block
if __name__ == "__main__":
    # Define dummy vocabulary and field objects for testing.
    class DummyVocab:
        def __init__(self):
            # Simple mapping for demonstration
            self.stoi = {
                "<s>": 0,
                "</s>": 1,
                "hello": 2,
                "world": 3,
                "how": 4,
                "are": 5,
                "you": 6,
            }
            self.itos = {i: s for s, i in self.stoi.items()}

    class DummyField:
        def __init__(self):
            self.init_token = "<s>"
            self.eos_token = "</s>"
            self.vocab = DummyVocab()

    src_field = DummyField()
    trg_field = DummyField()

    # Hyperparameters for dummy model
    INPUT_DIM = len(src_field.vocab.stoi)
    OUTPUT_DIM = len(trg_field.vocab.stoi)
    EMB_DIM = 16
    HID_DIM = 32
    NUM_LAYERS = 1
    DROPOUT = 0.1

    # Instantiate encoder, decoder, and the seq2seq model
    encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT)
    decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, NUM_LAYERS, DROPOUT)
    model = LSTMModel(encoder, decoder)

    # Test the translate() method with a dummy input
    sample_input = "hello world"
    print("Input:", sample_input)
    translation = model.translate(sample_input, src_field, trg_field)
    print("Translation:", translation)


Input: hello world
Translation: hello hello world how how world how how world how how world how how world how how world how how world how how world how how world how how world how how world how how world how how world how how world how how world how how world how


# not in other code! mix of edits and attempts to circumnavigate file directory shenanigans

In [141]:
old = list()
new = list()
for og, nog in pairs:
    old.append(og)
    new.append(nog)

In [38]:
import json
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
import random

# constants
LEMMATIZER = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))
FILENAME = 'movie_plots.json'

def preprocess_sentence(sentence:str, 
                        lemmatizer: WordNetLemmatizer = LEMMATIZER, 
                        stop_words: set = STOP_WORDS) -> str:
    """
    This function applies various text-normalization techniques to a sentence that is provided.
    Args:
        sentence (str): The sentence upon which text normalization must be processed.
        lemmatizer (WordNetLemmatizer): The lemmatizer for root words.
        stop_words (str): The stop words that must be removed.
    Returns:
        str: A normalized sentence string.
    """
    # Apply case-folding on your text.
    sentence = sentence.lower()
    
    # Remove any punctuations within your sentence.
    sentence = sentence.translate(str.maketrans('', '', string.punctuation))
    
    tokens = word_tokenize(sentence)
  
    # Remove stop words and lemmatize your sentence if they are provided
    # @TAs — changed
    if stop_words is not None:
        tokens = [word for word in tokens if word not in stop_words]
    if lemmatizer is not None:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    preprocessed = ' '.join(tokens)
    
    return preprocessed

In [142]:
for i in range(len(old)):
    old[i] = preprocess_sentence(old[i].replace(',', ' ').replace(';', ' ').replace(':', ' '))

In [143]:
for i in range(len(new)):
    new[i] = preprocess_sentence(new[i].replace(',', '').replace(';', '').replace(':', '').replace('—', ' '))

In [225]:
# new

In [44]:
from collections import Counter

In [93]:
# " ".join(old).split()

In [159]:
counted = Counter(" ".join(old).split())
len(counted.keys())

2731

In [94]:
# " ".join(new).split()

In [160]:
counted = Counter(" ".join(new).split())
len(counted.keys())

2217

In [20]:
pairs = load_parallel_dataset('shakez.csv')
test_size = 0.1
val_size = 0.1
train_pairs, val_pairs, test_pairs = split_data(pairs, test_size=test_size, val_size=val_size)

In [95]:
pairs = list(zip(old, new))

In [104]:
from transformers import BertTokenizerFast

In [122]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [125]:
tokenizer.vocab_size

30522

In [134]:
debug = 0
def main():


    encoder_params = (2731, 128, CONFIG['lstm_hidden_size'], CONFIG['lstm_num_layers'], CONFIG['dropout'])
    decoder_params = (2224, 128, CONFIG['lstm_hidden_size'], CONFIG['lstm_num_layers'], CONFIG['dropout'])

    # Initialize the LSTM Seq2Seq model
    model = LSTMModel(encoder=Encoder(*encoder_params), decoder=Decoder(*decoder_params))
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    # Utilize the tokenizer on the inputted data
    custom_tokens = [w for w in custom_words if w not in tokenizer.get_vocab()]
    tokenizer.add_tokens(custom_tokens)


    # Load dataset and create DataLoader objects
    # train_pairs, val_pairs, test_pairs = load_data(CONFIG)
    train_loader, val_loader, test_loader = create_dataloaders(
        train_pairs,
        val_pairs,
        test_pairs,
        tokenizer=tokenizer,
        max_length=CONFIG["transformer_max_length"],
        batch_size=CONFIG["batch_size"],
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])

    # Training loop
    for epoch in range(CONFIG["num_epochs"]):
        model.train()
        total_loss = 0
        for batch in train_loader:
            # print(batch.keys())
            optimizer.zero_grad()
            loss = model(**batch)  # Assume the model returns a loss given a batch
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} - Training Loss: {avg_loss:.4f}")

        samples = [(src, ref, model.translate(src)) for src, ref in val_pairs[:5]]
        # Evaluate on validation set
        model.eval()
        bleu_score = bleu(model, val_pairs)
        chrf_score = chrf(model, val_pairs)
        print(f"Epoch {epoch+1} - Validation BLEU Score: {bleu_score:.2f}")
        print(f"Epoch {epoch+1} - Validation chrF Score: {chrf_score:.2f}")

        params = {
            "model": "lstm",
            "dataset": CONFIG["dataset"],
            "lr": CONFIG["learning_rate"],
            "batch_size": CONFIG["batch_size"],
            "epochs": CONFIG["num_epochs"],
        }

        save_results(
            "results/lstm.csv",
            params=params,
            metrics={"bleu": bleu_score, "chrf": chrf_score},
            samples=samples,
            extras={
                "train_loss": avg_loss,
            },
        )

    # Generate sample outputs on test data
    model.eval()
    print("\nSample Translations:")
    for source, reference in test_pairs[:5]:
        translation = model.translate(source)
        print("Input:      ", source)
        print("Reference:  ", reference)
        print("Translation:", translation)
        print("-" * 50)



# attempt with new code

In [145]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [146]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers, dropout=dropout, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

In [147]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers, dropout):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)  # because LSTM expects [batch, seq_len, features]
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

In [269]:
# LSTM Class
class LSTM(nn.Module):
    def __init__(self, encoder, decoder):
        super(LSTM, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size)
        hidden, cell = self.encoder(src)
        input = trg[:, 0]  # <sos> token

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            top1 = output.argmax(1)
            input = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else top1

        return outputs

    def translate(self, sentence, src_vocab, trg_vocab, max_len=50):
        self.eval()

        tokens = ['<sos>'] + src_vocab.tokenize(sentence) + ['<eos>']
        numericalized = [src_vocab.stoi.get(tok, src_vocab.unk_idx) for tok in tokens]
        tensor = torch.LongTensor(numericalized).unsqueeze(0)

        with torch.no_grad():
            hidden, cell = self.encoder(tensor)

        trg_indexes = [trg_vocab.sos_idx]

        for i in range(max_len):
            trg_tensor = torch.LongTensor([trg_indexes[-1]])

            with torch.no_grad():
                output, hidden, cell = self.decoder(trg_tensor, hidden, cell)

            prob = F.softmax(output, dim=0)
            pred_token = torch.multinomial(prob, 1).item()
            if pred_token > len(trg_vocab):
                pred_token = trg_vocab.stoi('<UNK>')
            trg_indexes.append(pred_token)

            if pred_token == trg_vocab.eos_idx and not(i < (len(sentence.split()))):
                    break

        translated_tokens = [trg_vocab.itos[i] for i in trg_indexes[1:] if i != trg_vocab.eos_idx]
        return ' '.join(translated_tokens)

In [183]:
# dataset fxns
from torch.utils.data import Dataset, DataLoader
import torch

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab, max_len=50):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, trg = self.pairs[idx]
        src_ids = self.pad_sequence(self.src_vocab.numericalize(src))
        trg_ids = self.pad_sequence(self.trg_vocab.numericalize(trg))
        return torch.tensor(src_ids), torch.tensor(trg_ids)

    def pad_sequence(self, seq):
        seq = seq[:self.max_len]
        seq += [self.src_vocab.pad_idx] * (self.max_len - len(seq))
        return seq

In [209]:
# training functions
import torch.optim as optim

def train(model, dataloader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    batch_counter = 0

    for src, trg in dataloader:
        batch_counter += 1
        if batch_counter % 10 == 0:
            print('Currently on Batch: ', batch_counter)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]

        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


def evaluate(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in dataloader:
            output = model(src, trg, 0)  # Turn off teacher forcing
            output_dim = output.shape[-1]

            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


In [241]:
# custom tokenizer
from collections import Counter
import re

class Vocab:
    def __init__(self, texts):
        self.counter = Counter()
        for text in texts:
            tokens = self.tokenize(text)
            self.counter.update(tokens)

        self.tokens = [tok for tok in self.counter.keys()] + ['<padding>', '<UNK>', '<s>', '</s>']
        self.stoi = {tok: i for i, tok in enumerate(self.tokens)}
        self.itos = {i: tok for tok, i in self.stoi.items()}
        self.pad_idx = self.stoi['<padding>']
        self.unk_idx = self.stoi['<UNK>']
        self.sos_idx = self.stoi['<s>']
        self.eos_idx = self.stoi['</s>']

    def tokenize(self, text):
        # Basic tokenizer, space split and punctuation separation
        text = re.sub(r"([.,!?;])", r" \1", text.lower())
        return text.strip().split()

    def numericalize(self, text):
        tokens = ['<s>'] + self.tokenize(text) + ['</s>']
        return [self.stoi.get(tok, '<UNK>') for tok in tokens]

    def denumericalize(self, indices):
        return ' '.join([self.itos.get(idx, '<UNK>') for idx in indices if idx != self.pad_idx])

In [271]:


# Prepare vocabularies
src_vocab = Vocab([src for src, _ in pairs])
trg_vocab = Vocab([trg for _, trg in pairs])

INPUT_DIM = len(src_vocab.stoi)
OUTPUT_DIM = len(src_vocab.stoi)
EMB_DIM = 128
HID_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
model = LSTM(encoder, decoder)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

dataset = TranslationDataset(pairs, src_vocab, trg_vocab)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


In [272]:
for epoch in range(10):
    train_loss = train(model, dataloader, optimizer, criterion)
    val_loss = evaluate(model, dataloader, criterion)
    print(f'Epoch {epoch+1} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')


Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 1 | Train Loss: 1.838 | Val Loss: 1.132
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 2 | Train Loss: 1.087 | Val Loss: 1.146
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 3 | Train Loss: 1.096 | Val Loss: 1.118
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 4 | Train Loss: 1.029 | Val Loss: 1.168
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 5 | Train Loss: 1.015 | Val Loss: 1.141
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 6 | Train Loss: 1.006 | Val Loss: 1.166
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 7 | Train Loss: 1.002 | Val Loss: 1.110
Currently on Batch:  10
Currently on Batch:  20
Currently on Batch:  30
Epoch 8 | Train Loss: 0.988 | Val Loss: 1.121
Currently on Batch:  10
Currently on Batch:  20
Currentl

In [273]:
model.translate(to_translate, src_vocab, trg_vocab)

TypeError: object of type 'Vocab' has no len()

In [255]:
pairs[0]

('brother help thy fainting hand fear hath made thee faint hath fell devouring receptacle hateful cocytus misty mouth',
 'yo bro pas shaky hand fear got bailing like ditch gross savage maw')

In [258]:
to_translate =  pairs[0][0]

In [ ]:
def main():


    encoder_params = (2731, 128, CONFIG['lstm_hidden_size'], CONFIG['lstm_num_layers'], CONFIG['dropout'])
    decoder_params = (2224, 128, CONFIG['lstm_hidden_size'], CONFIG['lstm_num_layers'], CONFIG['dropout'])

    # Initialize the LSTM Seq2Seq model
    model = LSTMModel(encoder=Encoder(*encoder_params), decoder=Decoder(*decoder_params))
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

    # Utilize the tokenizer on the inputted data
    custom_tokens = [w for w in custom_words if w not in tokenizer.get_vocab()]
    tokenizer.add_tokens(custom_tokens)


    # Load dataset and create DataLoader objects
    # train_pairs, val_pairs, test_pairs = load_data(CONFIG)
    train_loader, val_loader, test_loader = create_dataloaders(
        train_pairs,
        val_pairs,
        test_pairs,
        tokenizer=tokenizer,
        max_length=CONFIG["transformer_max_length"],
        batch_size=CONFIG["batch_size"],
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])

    # Training loop
    for epoch in range(CONFIG["num_epochs"]):
        model.train()
        total_loss = 0
        for batch in train_loader:
            # print(batch.keys())
            optimizer.zero_grad()
            loss = model(**batch)  # Assume the model returns a loss given a batch
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} - Training Loss: {avg_loss:.4f}")

        samples = [(src, ref, model.translate(src)) for src, ref in val_pairs[:5]]
        # Evaluate on validation set
        model.eval()
        bleu_score = bleu(model, val_pairs)
        chrf_score = chrf(model, val_pairs)
        print(f"Epoch {epoch+1} - Validation BLEU Score: {bleu_score:.2f}")
        print(f"Epoch {epoch+1} - Validation chrF Score: {chrf_score:.2f}")

        params = {
            "model": "lstm",
            "dataset": CONFIG["dataset"],
            "lr": CONFIG["learning_rate"],
            "batch_size": CONFIG["batch_size"],
            "epochs": CONFIG["num_epochs"],
        }

        save_results(
            "results/lstm.csv",
            params=params,
            metrics={"bleu": bleu_score, "chrf": chrf_score},
            samples=samples,
            extras={
                "train_loss": avg_loss,
            },
        )

    # Generate sample outputs on test data
    model.eval()
    print("\nSample Translations:")
    for source, reference in test_pairs[:5]:
        translation = model.translate(source)
        print("Input:      ", source)
        print("Reference:  ", reference)
        print("Translation:", translation)
        print("-" * 50)